In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots

df = load_all_snapshots()

cols = ["pfx_x", "pfx_z", "release_speed", "release_spin_rate",
        "release_extension", "release_pos_x", "release_pos_z", "p_throws"]
print(df[cols].dtypes)
print()
print(df[cols].isna().mean().round(4))
print()
print(df[cols].describe().round(2).to_string())

pfx_x                Float64
pfx_z                Float64
release_speed        Float64
release_spin_rate      Int64
release_extension    Float64
release_pos_x        Float64
release_pos_z        Float64
p_throws                 str
dtype: object

pfx_x                0.0036
pfx_z                0.0036
release_speed        0.0036
release_spin_rate    0.0084
release_extension    0.0050
release_pos_x        0.0036
release_pos_z        0.0036
p_throws             0.0000
dtype: float64

          pfx_x     pfx_z  release_speed  release_spin_rate  release_extension  release_pos_x  release_pos_z
count  708058.0  708058.0       708061.0           704659.0           707057.0       708058.0       708058.0
mean       -0.1      0.59          89.15             2256.0               6.46          -0.83           5.76
std         0.9      0.71           5.98             362.04               0.46           1.87           0.53
min       -2.95     -2.45           31.9               14.0                3.

In [2]:
mv = (
    df[df["pitch_type"].notna()]
    .groupby(["p_throws", "pitch_type"])
    .agg(
        n=("pfx_x", "size"),
        pfx_x=("pfx_x", "mean"),
        pfx_z=("pfx_z", "mean"),
        velo=("release_speed", "mean"),
        spin=("release_spin_rate", "mean"),
    )
    .round(2)
)
print(mv[mv["n"] >= 1000].to_string())

                          n  pfx_x  pfx_z   velo     spin
p_throws pitch_type                                      
L        CH           26095   1.18   0.49  84.22   1762.6
         CU           12693  -0.65  -0.75  78.61   2461.6
         FC           13177  -0.17   0.59  87.88  2307.86
         FF           62473   0.68   1.31  93.27  2267.12
         FS            2003   0.71   0.54  84.66   1167.0
         KC            2900  -0.28  -0.62  80.83  2289.01
         SI           33223   1.26   0.68  92.63  2128.02
         SL           23421  -0.38   0.13  84.54  2380.29
         ST           12083  -1.16   0.08  80.38  2477.96
         SV            1656  -0.94  -0.45  79.99  2320.83
R        CH           46294  -1.18   0.41  86.15  1826.45
         CU           30955   0.81  -0.84  79.68  2626.67
         FC           44906   0.22   0.69  89.96  2421.05
         FF          163045  -0.63   1.31  94.68  2309.32
         FS           19869  -0.89   0.21  86.71  1329.35
         KC   

In [3]:
work = df[df["pitch_type"].notna()].copy()
work["pfx_x_arm"] = np.where(work["p_throws"] == "L",
                              -work["pfx_x"], work["pfx_x"])

norm = (
    work.groupby(["p_throws", "pitch_type"])
    .agg(n=("pfx_x_arm", "size"), pfx_x_arm=("pfx_x_arm", "mean"),
         pfx_z=("pfx_z", "mean"))
    .round(2)
)
print(norm[norm["n"] >= 1000].to_string())

                          n  pfx_x_arm  pfx_z
p_throws pitch_type                          
L        CH           26095      -1.18   0.49
         CU           12693       0.65  -0.75
         FC           13177       0.17   0.59
         FF           62473      -0.68   1.31
         FS            2003      -0.71   0.54
         KC            2900       0.28  -0.62
         SI           33223      -1.26   0.68
         SL           23421       0.38   0.13
         ST           12083       1.16   0.08
         SV            1656       0.94  -0.45
R        CH           46294      -1.18   0.41
         CU           30955       0.81  -0.84
         FC           44906       0.22   0.69
         FF          163045      -0.63   1.31
         FS           19869      -0.89   0.21
         KC            9449       0.68  -0.86
         SI           78471      -1.24    0.6
         SL           80692       0.38   0.14
         ST           39839       1.16   0.11
         SV            2039       

In [4]:
MIN_PITCHES_PER_TYPE = 50

arsenal = (
    work.groupby(["pitcher", "pitch_type"])
    .agg(
        n=("release_speed", "size"),
        velo=("release_speed", "mean"),
        pfx_x=("pfx_x_arm", "mean"),
        pfx_z=("pfx_z", "mean"),
        spin=("release_spin_rate", "mean"),
        ext=("release_extension", "mean"),
    )
)
arsenal = arsenal[arsenal["n"] >= MIN_PITCHES_PER_TYPE]

totals = work.groupby("pitcher").size().rename("total")
arsenal = arsenal.join(totals, on="pitcher")
arsenal["usage"] = arsenal["n"] / arsenal["total"]

print(f"{arsenal.index.get_level_values(0).nunique()} pitchers")
print(arsenal.head(15).round(2).to_string())

702 pitchers
                       n   velo  pfx_x  pfx_z     spin   ext  total  usage
pitcher pitch_type                                                        
434378  CH           160   84.0  -1.12   0.81  1779.49  5.97   1580    0.1
        CU           346  77.62   0.60  -1.12  2685.42  5.96   1580   0.22
        FF           767  93.47  -0.72   1.61  2395.02  5.94   1580   0.49
        SL           305   86.7   0.33   0.49  2441.64  5.97   1580   0.19
445276  FC           674  92.23   0.50   1.45  2629.43  6.76    801   0.84
        SI            65  93.21  -0.46   1.65  2267.53  6.83    801   0.08
445926  CH           140  85.19  -1.34   0.18  1882.21  6.37   1044   0.13
        CU            73  75.66   1.31  -0.69  2558.22  6.33   1044   0.07
        FC           533  88.81  -0.16   1.04  2200.57  6.25   1044   0.51
        SI           281  90.74  -1.33   0.63  1988.36  6.27   1044   0.27
450203  CH           307  84.95  -1.43   0.01  2172.43  6.29   2791   0.11
        CU  

In [5]:
depth = (
    arsenal.groupby("pitcher")
    .agg(n_pitches=("usage", "size"), total=("total", "first"))
)
depth = depth[depth["total"] >= 500]
print(depth["n_pitches"].value_counts().sort_index())
print()
print("mean arsenal size:", depth["n_pitches"].mean().round(2))

n_pitches
2     44
3    122
4    154
5    101
6     44
7      5
8      2
9      1
Name: count, dtype: Int64

mean arsenal size: 4.01


In [6]:
from src.data.player_ids import load_player_ids, display_name

def separation(pitcher_id):
    """Movement and velocity gap between a pitcher's primary fastball
    and each secondary pitch."""
    a = arsenal.loc[pitcher_id]
    fastballs = [p for p in ["FF", "SI", "FC"] if p in a.index]
    if not fastballs:
        return None
    primary = a.loc[fastballs].sort_values("usage", ascending=False).index[0]
    base = a.loc[primary]

    rows = []
    for pt, r in a.iterrows():
        if pt == primary:
            continue
        rows.append({
            "vs": primary, "pitch": pt, "usage": r["usage"],
            "velo_gap": base["velo"] - r["velo"],
            "move_gap": np.hypot(base["pfx_x"] - r["pfx_x"],
                                 base["pfx_z"] - r["pfx_z"]),
        })
    return pd.DataFrame(rows)

ids = load_player_ids(depth.index.tolist())
names = display_name(ids)

example = depth.nlargest(5, "total").index
for pid in example:
    print(f"--- {names.get(pid, pid)} ---")
    print(separation(pid).round(2).to_string(index=False))
    print()

--- Webb, Logan ---
vs pitch  usage  velo_gap  move_gap
SI    CH   0.31      5.22      0.66
SI    FC   0.03      1.58      1.29
SI    FF   0.05     -0.03      0.87
SI    ST   0.21      8.81      2.63

--- Nola, Aaron ---
vs pitch  usage  velo_gap  move_gap
FF    CH   0.10      6.78      0.99
FF    FC   0.09      5.00      1.03
FF    KC   0.33     13.39      2.98
FF    SI   0.21      0.96      0.65

--- Cease, Dylan ---
vs pitch  usage  velo_gap  move_gap
FF    KC   0.08     14.57      2.89
FF    SL   0.43      9.21      1.43
FF    ST   0.04     14.26      2.71

--- Wheeler, Zack ---
vs pitch  usage  velo_gap  move_gap
FF    CU   0.10     13.48      2.48
FF    FC   0.10      4.12      1.01
FF    FS   0.07      9.12      0.82
FF    SI   0.19      0.63      0.92
FF    ST   0.13     10.31      1.84

--- Ragans, Cole ---
vs pitch  usage  velo_gap  move_gap
FF    CH   0.24     10.61      0.51
FF    FC   0.11      4.64      1.16
FF    KC   0.10     14.64      3.14
FF    SL   0.13      9.41   